# OE3: reproducción autónoma en Google Colab

**Objetivo oficial (literal).** Determinar el efecto de la configuración de los roles de los agentes, los protocolos de comunicación y los mecanismos de inteligencia de enjambre sobre el tiempo de convergencia y la eficiencia de la adaptación de contenidos multimodales.

**Qué comprueba este cuaderno.** Reejecuta el análisis pre-registrado (49 contrastes de efectos principales, ANOVA factorial con η² parcial, Friedman, deriva, sensibilidad y resumen del entorno) desde las observaciones de los 19 200 ciclos y lo compara con el resultado local.

**Datos de entrada.** `raw_observations.csv` (19 200 ciclos), `batches.json`, `environment.json`, `provenance.json`, monitor de recursos y el script de análisis, con el resultado local para comparar.

**Qué no hace.** No recalcula los tiempos medidos, que son los del equipo local con interferencia registrada. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Pulse «Ejecutar todo» en un entorno nuevo de Colab: los datos se descargan solos desde GitHub (commit fijado) y se verifican por SHA-256; no hay que subir ni buscar archivos.

**Contenido.** 1 descripción · 2 entorno y código (clonado de la etiqueta inmutable) · 3 descarga automática y verificación SHA-256 · 4 validación de integridad y estructura · 5 preprocesamiento y cálculo de indicadores (scripts sellados) · 6 análisis estadístico · 7 comparación con el resultado local y con las cifras de la tesis · 8 tablas · 9 gráficos · 10 pruebas automáticas · 11 resumen. Las diferencias se clasifican como exacta, dentro de tolerancia, explicada por versión, no explicada o no reproducible.

## 1. Entorno y código de análisis
El código vive en el repositorio público (módulos legibles en `reproducibility/etapa26_colab_autonomo/src/repro/`); los scripts de análisis sellados vienen dentro de los paquetes de datos, con su hash verificado.

In [ ]:
import os, sys, platform, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Renato5Lara/multiagent-llm-education.git'
REPO_REF = 'repro-etapa26-v2'            # etiqueta inmutable del codigo; los datos estan fijados por commit en manifests/data_manifest.json
if os.environ.get('REPRO_LOCAL_REPO'):   # solo para pruebas locales antes de publicar la etiqueta
    REPO_DIR = os.environ['REPRO_LOCAL_REPO']
else:
    REPO_DIR = 'repo_repro'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch', REPO_REF, '--filter=blob:none', '--sparse', REPO_URL, REPO_DIR], check=True)
        subprocess.run(['git', '-C', REPO_DIR, 'sparse-checkout', 'set', 'reproducibility/etapa26_colab_autonomo'], check=True)   # solo el codigo de esta etapa
sys.path.insert(0, os.path.join(REPO_DIR, 'reproducibility', 'etapa26_colab_autonomo', 'src'))
import repro
from repro import sh
print('Codigo de analisis:', REPO_URL, '@', REPO_REF)


In [ ]:
try:
    import statsmodels
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'statsmodels'], check=True)
    import statsmodels
import statsmodels, numpy, scipy, pandas, matplotlib
ENTORNO = dict(python=platform.python_version(), plataforma=platform.platform(), numpy=numpy.__version__, scipy=scipy.__version__, pandas=pandas.__version__, matplotlib=matplotlib.__version__, cpu=os.cpu_count(), statsmodels=statsmodels.__version__)
print(ENTORNO)


## 2. Descarga automática y verificación de los paquetes (SHA-256)

In [ ]:
# Descarga automatica desde GitHub (commit fijado) y verificacion SHA-256 contra el manifiesto. No hay que subir nada.
# Si falta un archivo, no se puede descargar o su hash no coincide, se detiene con el archivo, la URL y el hash esperado.
PAQUETES = ['oe3_colab_paquete_v2']
repro.fetch_packages(PAQUETES, dest='oe3')


## 3. Verificación interna, cálculo y análisis (celdas del notebook de comprobación publicado, sin cambios salvo la ejecución de scripts)

In [ ]:
import hashlib
base='oe3/campana_01'; bad=[]; checked=[]
for l in open(base+'/checksums.txt'):
    l=l.strip()
    if not l: continue
    h,n=l.split(None,1); n=n.strip().lstrip('*'); p=os.path.join(base,n)
    if os.path.exists(p):
        checked.append(n)
        if hashlib.sha256(open(p,'rb').read()).hexdigest()!=h: bad.append(n)
print('verificados:', checked); print('hash distinto:', bad); assert not bad

In [ ]:
sh(f"python oe3/analisis_oe3.py oe3/campana_01/raw_observations.csv oe3/salida_colab oe3/campana_01_monitor.csv > oe3/log_analisis.txt 2>&1")
print(open('oe3/log_analisis.txt').read()[-600:])

In [ ]:
import json
a=json.load(open('oe3/salida_colab/resultados_oe3.json')); b=json.load(open('oe3/esperado_resultados_oe3_local.json'))
print('verificaciones iguales:', a['verificaciones']==b['verificaciones'], a['verificaciones']['observaciones'], 'observaciones')
mx=0
for ca,cb in zip(a['contrastes'],b['contrastes']):
    assert (ca['respuesta'],ca['factor'],ca['contraste'])==(cb['respuesta'],cb['factor'],cb['contraste'])
    mx=max(mx,abs(ca['estimacion']-cb['estimacion']),abs(ca['ic95'][0]-cb['ic95'][0]),abs(ca['ic95'][1]-cb['ic95'][1]))
print('contrastes comparados:', len(a['contrastes']), '| diferencia maxima en estimaciones e IC:', mx)
mp=max(abs(ca['p_holm']-cb['p_holm']) for ca,cb in zip(a['contrastes'],b['contrastes']))
print('diferencia maxima en p de Holm:', mp)
for k in a['anova_factorial']:
    ea=[(r['termino'],round(r['eta2_parcial'],6)) for r in a['anova_factorial'][k][:6]]; eb=[(r['termino'],round(r['eta2_parcial'],6)) for r in b['anova_factorial'][k][:6]]
    print(k, 'eta2 parcial (6 mayores) iguales:', ea==eb)
print('Friedman iguales:', [round(x['p'],12) for x in a['friedman_tconv']]==[round(x['p'],12) for x in b['friedman_tconv']])
print('despacho/lotes (T_conv):', [ (c['estimacion'],c['ic95']) for c in a['contrastes'] if c['respuesta']=='log_tconv' and c['factor']=='P1_despacho'])
print('sensibilidad:', a['sensibilidad']['despacho_sin_cognitivo_razon_tconv'], b['sensibilidad']['despacho_sin_cognitivo_razon_tconv'])
print('deriva Spearman:', a['deriva']['spearman_residuo_vs_orden'][0], b['deriva']['spearman_residuo_vs_orden'][0])
print('entorno:', a['entorno']['fraccion_con_procesos_ajenos_gt10'], b['entorno']['fraccion_con_procesos_ajenos_gt10'])

## 4. Comparación con las cifras de la tesis
Cada cifra se lee del resultado **reproducido** y se compara con la que figura en la tesis (versión CIERRE_FINAL) y en el JSON oficial sellado (`manifests/valores_oficiales.json`).

In [ ]:
from repro import official, validation, tables, figures
from repro.data_loading import sha256_file
import pandas as pd
RUTAS = {'resultados_oe3.json': 'oe3/salida_colab/resultados_oe3.json', 'contrastes_efectos_principales.csv': 'oe3/salida_colab/contrastes_efectos_principales.csv'}
cifras = official.evaluar('OE3', RUTAS)
print(cifras[['id', 'descripcion', 'esperado', 'obtenido', 'dif', 'estado']].to_string(index=False))


## 5. Clasificación de las diferencias frente al resultado local
Exacta · dentro de tolerancia (1e-9) · explicada por versión · no explicada · no reproducible.

In [ ]:
import scipy
REG = validation.Registro(scipy_version=scipy.__version__)
COMPARACIONES = [('OE3 resultados_oe3.json', 'oe3/salida_colab/resultados_oe3.json', 'oe3/esperado_resultados_oe3_local.json')]
for _e, _a, _b in COMPARACIONES:
    REG.comparar(_e, _a, _b)


## 6. Tablas

In [ ]:
OUT = 'outputs'
TABLAS = {'contrastes_efectos_principales': 'oe3/salida_colab/contrastes_efectos_principales.csv'}
tablas_generadas = tables.exportar_tablas('OE3', TABLAS, os.path.join(OUT, 'tables'))
print('Tablas generadas:')
for _t in tablas_generadas:
    print('  ', _t)


## 7. Gráficos
Se generan desde la salida del análisis ejecutado arriba (no desde valores copiados). Se exportan PNG (300 ppi), SVG y el CSV de la serie dibujada (`figure_data/`).

In [ ]:
METAS = official.cargar_valores()['metas_de_diseno']
_a = json.load(open('oe3/esperado_resultados_oe3_local.json', encoding='utf8'))
pd.DataFrame([{**c, 'ic95_inf': c['ic95'][0], 'ic95_sup': c['ic95'][1]} for c in _a['contrastes']]).drop(columns='ic95').to_csv('oe3/esperado_contrastes.csv', index=False)
fig = figures.fig_oe3(RUTAS['contrastes_efectos_principales.csv'], os.path.join(OUT, 'figures', 'OE3'), metas=METAS)
ref = figures.fig_oe3('oe3/esperado_contrastes.csv', os.path.join(OUT, 'figures_ref_desde_resultado_local', 'OE3'), metas=METAS)
REG.comparar('Serie de la Figura 3 reproducida frente a la serie dibujada desde el resultado oficial local', fig['serie_csv'], ref['serie_csv'])
_t = official.cargar_valores()['figuras_tesis']['fig_oe3']
IDENTICA = sha256_file(fig['png']) == _t['sha256_png_tesis']
print('Figura 3: PNG identico bit a bit al de la tesis (pagina %s del indice): %s' % (_t['pagina_indice'], IDENTICA))
if not IDENTICA:
    print('  Esperable fuera del entorno de la tesis (matplotlib 3.10.3 y fuente Arial): la imagen puede diferir en pixeles; la comprobacion valida es la de la serie dibujada.')
try:
    from IPython.display import Image, display
    display(Image(fig['png']))
except Exception:
    pass


## 8. Pruebas automáticas de consistencia

In [ ]:
fallos = []
if cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE']).any():
    fallos.append('cifras oficiales: ' + str(cifras[cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE'])][['id', 'estado']].values.tolist()))
peor, cuenta = REG.veredicto()
if peor in ('NO_EXPLICADA', 'NO_REPRODUCIBLE'):
    fallos.append('comparacion con el resultado local: ' + peor)
if not (os.path.getsize(fig['png']) > 0 and os.path.getsize(fig['serie_csv']) > 0):
    fallos.append('figura vacia')
assert not fallos, fallos
print('PRUEBAS AUTOMATICAS DE CONSISTENCIA: todas superadas | cifras oficiales comprobadas:', len(cifras), '| comparaciones:', cuenta)


## 9. Resumen final de la reproducción

In [ ]:
import shutil
resumen = tables.guardar_resumen('OE3', os.path.join(OUT, 'reports'), REG.tabla(), cifras, ENTORNO, extra=dict(veredicto=peor, conteo=cuenta, paquetes=PAQUETES))
print('RESUMEN DE REPRODUCCION OE3')
print(REG.tabla()[['comparacion', 'categoria', 'dif_abs_max']].to_string(index=False))
print('Veredicto (peor categoria):', peor, cuenta)
print('Cifras de la tesis comprobadas: %d (%d coinciden exactamente, %d dentro del redondeo)' % (len(cifras), (cifras.estado == 'COINCIDE').sum(), (cifras.estado == 'COINCIDE_REDONDEO').sum()))
print('LIMITACIONES: campana no controlada con interferencia ambiental; el p de Wilcoxon de «social frente a PSO» depende de la version de scipy (5,1e-4 entre 1.16.3 y 1.18.1) sin afectar a Holm ni a las conclusiones; la version 1 del notebook esta descartada.')
print('Alcance: se reproduce el analisis, no la adquisicion de datos; no se repite ninguna campana.')
_z = shutil.make_archive('resultados_OE3', 'zip', OUT)
print('Salidas empaquetadas en', os.path.basename(_z))
if IN_COLAB:
    from google.colab import files
    files.download(_z)
